# DNT Organoid AI — Google Colab QuantSeq 3′ mRNA-seq Real-Data Validation
## Lexogen QuantSeq Verification with Existing Google Drive STAR Reference Index

**Objective:** Execute genuine end-to-end RNA-seq quantification on public human iPSC dopaminergic neuron Lexogen QuantSeq 3′ mRNA-seq data (**GEO GSE291145**) using our verified **Ensembl GRCh38 Release 112 STAR genome index** stored in Google Drive.

### Computational & Storage Architecture
1. **Execution Runtime:** Google Colab High-RAM runtime (or GPU runtime with $\ge 35\text{ GB}$ system RAM and $\ge 70\text{ GB}$ local NVMe scratch in `/content`).
2. **Persistent Storage (Google Drive):** Accesses private archive `star_index_GRCh38_112_dense.tar` (28.2 GiB, SHA-256 `756ed0a395cd477db730f9f9a4199c77576da59fb5aab7169fa9a1a33dd56392`) via authorized user mount `drive.mount('/content/drive')`.
3. **Zero Redundant Generation:** Does **not** rebuild the index. Verifies the SHA-256 checksum and extracts the complete 16-file reference directly to `/content/references/star_index`.
4. **Genuine End-to-End Processing:** Runs `fastp` (QuantSeq poly(A) & adapter trimming) $\to$ `STAR` alignment $\to$ `RSeQC` empirical strandedness verification $\to$ `featureCounts` quantification (sense `-s 1`) without transcript length normalization.
5. **Downstream Integration:** Verifies raw integer counts directly feed DESeq2 median-of-ratios size-factor estimation without gene-length division.

### Step 1: Mount Google Drive & Environment Preflight Check
Authorize Google Drive to access your private storage folder containing `star_index_GRCh38_112_dense.tar`.

In [ ]:
import os
import sys
import shutil
from pathlib import Path

# Check hardware resources
total_ram_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3)
disk_free_gb = shutil.disk_usage('/content')[2] / (1024**3)

print(f"Detected System RAM: {total_ram_gb:.1f} GB")
print(f"Available Scratch Disk (/content): {disk_free_gb:.1f} GB")

# Strict Pre-flight Hardware Verification:
# Dense STAR index (~28.2 GiB archive) requires ~30 GB memory during alignment
# and ~40-50 GB scratch disk for extraction + BAM outputs.
if total_ram_gb < 28.0:
    raise RuntimeError(
        f"INSUFFICIENT RAM: Detected {total_ram_gb:.1f} GB RAM, but at least 30 GB is required "
        "to load the human GRCh38.112 dense STAR index without OOM crashes.\n"
        "ACTION: In Google Colab menu, go to: Runtime -> Change runtime type -> Choose 'High-RAM'."
    )

if disk_free_gb < 50.0:
    raise RuntimeError(
        f"INSUFFICIENT SCRATCH DISK: Detected {disk_free_gb:.1f} GB free in /content, but at least 50 GB "
        "is required to unpack the 28.2 GiB archive and store STAR BAM alignments.\n"
        "ACTION: Clean up existing /content scratch files or request a high-storage Colab runtime."
    )

print("Preflight Hardware Verification: PASSED (Sufficient RAM and Disk capacity confirmed).")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("Google Drive successfully mounted.")

### Step 2: Install Bioinformatics CLI Tools & Clone Repository
Install standard bioinformatics tools (`fastp`, `STAR`, `subread` for featureCounts, `rseqc`, `sra-tools`) via conda/apt, and clone the `feature/lexogen-quantseq-support` branch.

In [ ]:
# Install packages
!apt-get update -qq
!apt-get install -y -qq rna-star subread fastp sra-toolkit python3-pip
!pip install -q rseqc pandas numpy scipy pytest

# Verify tool availability
!STAR --version
!featureCounts -v
!fastp --version
!infer_experiment.py --version || echo "RSeQC installed"

# Clone DNT-Organoid-AI repository branch
%cd /content
!rm -rf DNT-Organoid-AI
!git clone -b feature/lexogen-quantseq-support https://github.com/bioprintingLabsInc/DNT-Organoid-AI.git
%cd /content/DNT-Organoid-AI
import sys
if '/content/DNT-Organoid-AI' not in sys.path:
    sys.path.insert(0, '/content/DNT-Organoid-AI')
print("Repository cloned and added to PYTHONPATH.")

### Step 3: Locate and Verify Google Drive STAR Reference Archive
Verifies the expected SHA-256 checksum (`756ed0a395cd477db730f9f9a4199c77576da59fb5aab7169fa9a1a33dd56392`) of `star_index_GRCh38_112_dense.tar` before extraction.

In [ ]:
import hashlib

# Search Google Drive for the archive
# When mounting Google Drive, files appear in /content/drive/MyDrive or /content/drive/Shareddrives
target_archive_name = 'star_index_GRCh38_112_dense.tar'
expected_sha256 = '756ed0a395cd477db730f9f9a4199c77576da59fb5aab7169fa9a1a33dd56392'

# Check common locations or search mounted Drive
search_roots = [
    Path('/content/drive/MyDrive'),
    Path('/content/drive/Shareddrives'),
    Path('/content/drive'),
]

archive_path = None
for root in search_roots:
    if root.exists():
        matches = list(root.rglob(target_archive_name))
        if matches:
            archive_path = matches[0]
            break

if archive_path is None or not archive_path.exists():
    # If the folder was shared with you, add a shortcut to 'My Drive' in the Google Drive web UI
    # (Right-click folder -> Organize -> Add shortcut -> 'My Drive')
    raise FileNotFoundError(
        f"Could not locate '{target_archive_name}' under mounted Google Drive (/content/drive)!\n"
        "TIP: If the folder '1lmkqtedPkGLNxwGYExrOJLq8kTiClW9J' is in 'Shared with me', please open Google Drive in your browser, "
        "right-click the folder, select 'Organize' -> 'Add shortcut', and choose 'My Drive'."
    )

print(f"Found STAR archive at: {archive_path}")
print(f"Archive size: {archive_path.stat().st_size / (1024**3):.2f} GiB")

# Checksum verification
print("Verifying SHA-256 checksum (streaming chunks)... Please wait.")
h = hashlib.sha256()
with open(archive_path, 'rb') as f:
    while chunk := f.read(16 * 1024 * 1024):
        h.update(chunk)
observed_sha256 = h.hexdigest()
print(f"Expected: {expected_sha256}")
print(f"Observed: {observed_sha256}")

assert observed_sha256.lower() == expected_sha256.lower(), "SHA-256 verification failed! Archive may be corrupted."
print("SHA-256 checksum MATCHES perfectly!")

### Step 4: Extract STAR Index to Fast Colab NVMe Local Storage
Unpacks the verified index directly onto `/content/references/star_index` to guarantee maximum IOPS during alignment, avoiding Drive FUSE limits.

In [ ]:
import tarfile
from src.quantification_bridge.star_runner import verify_star_index

local_index_dir = Path('/content/references/star_index')
local_index_dir.mkdir(parents=True, exist_ok=True)

# Check if already extracted
eval_res = verify_star_index(local_index_dir)
if eval_res['is_complete']:
    print("Local index already extracted and complete. Skipping extraction.")
else:
    print(f"Extracting {archive_path} to {local_index_dir}... This takes ~2-3 minutes on Colab NVMe.")
    with tarfile.open(archive_path, 'r:*') as tar:
        tar.extractall(path=local_index_dir)
    print("Extraction complete.")

# Verify all 14 required reference files
eval_res = verify_star_index(local_index_dir)
print(f"Index complete: {eval_res['is_complete']}")
print(f"Chromosomes: {eval_res['chromosome_count']}")
print(f"Total size: {eval_res['total_gb']:.2f} GiB")
print(f"STAR parameters: {eval_res['parameters']}")
assert eval_res['is_complete'], f"Missing required STAR files: {eval_res['missing_files']}"

### Step 5: Acquire Human GRCh38.112 GTF & BED12 Annotations
Downloads Ensembl Release 112 annotation GTF and generates the deterministic BED12 file required for RSeQC empirical strandedness verification.

In [ ]:
ref_dir = Path('/content/references')
gtf_gz = ref_dir / 'Homo_sapiens.GRCh38.112.gtf.gz'
gtf_path = ref_dir / 'Homo_sapiens.GRCh38.112.gtf'
bed12_path = ref_dir / 'Homo_sapiens.GRCh38.112.bed'

if not gtf_path.exists():
    print("Downloading Ensembl 112 GTF...")
    !wget -q -O /content/references/Homo_sapiens.GRCh38.112.gtf.gz https://ftp.ensembl.org/pub/release-112/gtf/homo_sapiens/Homo_sapiens.GRCh38.112.gtf.gz
    !gunzip -f /content/references/Homo_sapiens.GRCh38.112.gtf.gz

# Generate BED12 via gtfToGenePred and genePredToBed if missing
if not bed12_path.exists():
    print("Generating BED12 annotation for RSeQC...")
    !wget -q http://hgdownload.soe.ucsc.edu/admin/exe/linux.x86_64/gtfToGenePred -O /usr/local/bin/gtfToGenePred
    !wget -q http://hgdownload.soe.ucsc.edu/admin/exe/linux.x86_64/genePredToBed -O /usr/local/bin/genePredToBed
    !chmod +x /usr/local/bin/gtfToGenePred /usr/local/bin/genePredToBed
    !gtfToGenePred /content/references/Homo_sapiens.GRCh38.112.gtf /content/references/genes.genePred
    !genePredToBed /content/references/genes.genePred /content/references/Homo_sapiens.GRCh38.112.bed

print(f"GTF exists: {gtf_path.exists()} ({gtf_path.stat().st_size / (1024**2):.1f} MB)")
print(f"BED12 exists: {bed12_path.exists()} ({bed12_path.stat().st_size / (1024**2):.1f} MB)")

### Step 6: Acquire GSE291145 QuantSeq FASTQ Samples
Downloads real human iPSC dopaminergic neuron QuantSeq 3′ mRNA-seq reads for samples:
- `SRR32575599` (Control Rep 1, GSM8829727)
- `SRR32575598` (Control Rep 2, GSM8829728)
- `SRR32575595` (Treatment Rep 2, GSM8829731)

In [ ]:
data_dir = Path('/content/data/raw/real_quantseq')
data_dir.mkdir(parents=True, exist_ok=True)

samples_meta = [
    ("SRR32575599", "GSM8829727_CTRL_REP1"),
    ("SRR32575598", "GSM8829728_CTRL_REP2"),
    ("SRR32575595", "GSM8829731_TRT_REP2"),
]

for srr, label in samples_meta:
    fq_file = data_dir / f"{srr}.fastq.gz"
    if not fq_file.exists():
        print(f"Fetching 100,000 real reads for {srr} ({label})...")
        !fastq-dump -X 100000 --gzip -O /content/data/raw/real_quantseq {srr}
    print(f"{srr} ready: {fq_file.stat().st_size / (1024**2):.2f} MB")

### Step 7: Execute Universal Quantification Pipeline on QuantSeq Data
Executes the full pipeline with QuantSeq 3′ FWD configuration:
- **fastp**: Quality trimming, poly(A) tail clipping, adapter cutting.
- **STAR**: Single-end alignment to GRCh38.112 dense index, coordinate-sorted BAM.
- **RSeQC**: Empirical strandedness consensus (`infer_experiment.py`) verifying sense orientation.
- **featureCounts**: Sense-strand quantification (`-s 1`), counting reads per gene.

In [ ]:
import json
from src.quantification_bridge.library_type import LibraryType
from src.quantification_bridge.universal_pipeline import (
    UniversalFastqPipeline,
    UniversalPipelineConfig,
    ReferenceConfig,
    SampleInput,
)

out_dir = Path('/content/data/processed/quantseq_validation_output')
out_dir.mkdir(parents=True, exist_ok=True)

samples = [
    SampleInput(
        sample_id="GSM8829727_SRR32575599_CTRL_REP1",
        r1_path=data_dir / "SRR32575599.fastq.gz",
        read_layout="single_end",
        library_type=LibraryType.LEXOGEN_QUANTSEQ_3UTR_FWD,
        attributes={"condition": "control", "replicate": "1"},
    ),
    SampleInput(
        sample_id="GSM8829728_SRR32575598_CTRL_REP2",
        r1_path=data_dir / "SRR32575598.fastq.gz",
        read_layout="single_end",
        library_type=LibraryType.LEXOGEN_QUANTSEQ_3UTR_FWD,
        attributes={"condition": "control", "replicate": "2"},
    ),
    SampleInput(
        sample_id="GSM8829731_SRR32575595_TRT_REP2",
        r1_path=data_dir / "SRR32575595.fastq.gz",
        read_layout="single_end",
        library_type=LibraryType.LEXOGEN_QUANTSEQ_3UTR_FWD,
        attributes={"condition": "treatment", "replicate": "2"},
    ),
]

cfg = UniversalPipelineConfig(
    study_id="GSE291145_COLAB_VALIDATION",
    output_dir=out_dir,
    read_layout="single_end",
    library_type=LibraryType.LEXOGEN_QUANTSEQ_3UTR_FWD,
    threads=os.cpu_count() or 4,
    enforce_resources=False,
    require_approval=True,
)

ref_cfg = ReferenceConfig(
    species="Homo sapiens",
    genome_fasta=ref_dir / 'Homo_sapiens.GRCh38.dna.primary_assembly.fa',  # optional if star_index_dir exists
    gtf_annotation=gtf_path,
    star_index_dir=local_index_dir,
    bed12_annotation=bed12_path,
)

pipeline = UniversalFastqPipeline(config=cfg, references=ref_cfg, samples=samples)
plan = pipeline.plan()
print("Preflight valid:", plan.preflight.is_valid)
assert not plan.preflight.has_fatal_errors, f"Fatal preflight errors: {plan.preflight.findings}"

pipeline.approve(plan.approval_token)
print("Executing pipeline in Colab High-RAM runtime...")
result = pipeline.run(approval_token=plan.approval_token)
print(f"Pipeline Result Status: {result.status}")
assert result.status == "COMPLETED", f"Pipeline failed at {result.failed_stage}: {result.findings}"

### Step 8: Validate Gene-Count Matrix & Downstream DESeq2 Integration
Inspect the resulting raw count matrix, verify that no transcript length division occurred, and test DESeq2 median-of-ratios size-factor calculation.

In [ ]:
import pandas as pd
import numpy as np

counts_file = out_dir / 'counts' / 'raw_gene_counts.tsv'
counts_df = pd.read_csv(counts_file, sep='\t', index_col=0)

print("=== RAW GENE COUNT MATRIX (First 10 genes) ===")
print(counts_df.head(10))
print(f"\nTotal Ensembl Genes: {len(counts_df)}")
print(f"Genes detected (>0 counts across samples): {(counts_df.sum(axis=1) > 0).sum()}")
print(f"Genes with >= 10 counts: {(counts_df.sum(axis=1) >= 10).sum()}")

# Check DESeq2 median-of-ratios
log_counts = np.log(counts_df.replace(0, np.nan))
geo_mean = np.exp(log_counts.mean(axis=1))
valid_genes = geo_mean.dropna().index
ratios = counts_df.loc[valid_genes].div(geo_mean.loc[valid_genes], axis=0)
size_factors = ratios.median(axis=0).to_dict()

print("\n=== DESeq2 SIZE FACTORS ===")
for s, sf in size_factors.items():
    print(f"  {s}: {sf:.4f}")

assert all(sf > 0 for sf in size_factors.values()), "All size factors must be strictly positive!"
print("\nDownstream DESeq2 normalization test PASSED successfully!")

### Step 9: Save Validation Deliverables to Google Drive
Syncs the raw count matrix, summary QC metrics, RSeQC strandedness consensus, and execution log back to your persistent Google Drive folder.

In [ ]:
gdrive_dest_dir = archive_path.parent / 'quantseq_validation_results'
gdrive_dest_dir.mkdir(parents=True, exist_ok=True)

print(f"Exporting deliverables to Google Drive: {gdrive_dest_dir}")
# Copy counts, summary, and reports
shutil.copy(counts_file, gdrive_dest_dir / 'raw_gene_counts.tsv')
shutil.copy(out_dir / 'counts' / 'raw_gene_counts.tsv.summary', gdrive_dest_dir / 'raw_gene_counts.tsv.summary')
shutil.copy(out_dir / 'strandedness_evidence.json', gdrive_dest_dir / 'strandedness_evidence.json')

# Save structured summary JSON
summary_metrics = {
    "study_id": "GSE291145",
    "reference_index_archive": str(archive_path.name),
    "reference_index_sha256": expected_sha256,
    "library_type": "LEXOGEN_QUANTSEQ_3UTR_FWD",
    "total_genes_quantified": len(counts_df),
    "detected_genes_gt0": int((counts_df.sum(axis=1) > 0).sum()),
    "deseq2_size_factors": {k: round(float(v), 4) for k, v in size_factors.items()},
}
with open(gdrive_dest_dir / 'validation_metrics_summary.json', 'w') as f:
    json.dump(summary_metrics, f, indent=2)

print("Validation artifacts successfully saved to Google Drive!")